# [LEC] A2A 프로토콜의 이해

슬라이드와 설명, 퀴즈로 Agent가 다른 Agent에게 일을 맡기는 표준인 A2A를 익힙니다.

**목차**

- 1. Agent 2 Agent (슬라이드) (퀴즈)
  - 1.1 A2A 등장 배경 (슬라이드) (퀴즈)
  - 1.2 MCP와 A2A의 차이 (슬라이드) (퀴즈)
- 2. A2A 핵심 개념 (슬라이드) (퀴즈)
  - 2.1 Agent Card (슬라이드) (퀴즈)
  - 2.2 Task, Message, Artifact (슬라이드) (퀴즈)
- 3. A2A Server 구현 (슬라이드) (퀴즈)
  - 3.1 AgentExecutor (슬라이드) (퀴즈)
  - 3.2 AgentCard (슬라이드) (퀴즈)
  - 3.3 RequestHandler (슬라이드) (퀴즈)
- 4. A2A Client 구현 (슬라이드) (퀴즈)
  - 4.1 A2ACardResolver (슬라이드) (퀴즈)
  - 4.2 SendMessageRequest (슬라이드) (퀴즈)
- 5. 정리 퀴즈 (퀴즈)

In [ ]:
import sys

sys.path.insert(0, "..")  # 챕터 폴더의 assets 도구를 불러오기 위한 경로

from assets.widgets import show_quiz, show_slides, slide_setup

slide_setup()  # 슬라이드 뷰어(PDF.js)를 불러옵니다. 슬라이드가 비어 보이면 이 셀을 다시 실행합니다.

In [ ]:
show_slides("3-3", [1, 2])

이번 시간은 Agent가 다른 Agent에게 일을 맡기고 결과를 받는 표준인 A2A(Agent to Agent)를 봅니다.

1. Agent 2 Agent: A2A 등장 배경, MCP와 A2A의 차이
2. A2A 핵심 개념: Agent Card, Task·Message·Artifact
3. A2A Server 구현: AgentExecutor, AgentCard, RequestHandler
4. A2A Client 구현: A2ACardResolver, SendMessageRequest

## 1. Agent 2 Agent <small>(슬라이드) (퀴즈)</small>

### 1.1 A2A 등장 배경 <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("3-3", [3, 4])

A2A는 Agent가 다른 Agent에게 작업을 위임하고 결과를 주고받기 위한 통신 프로토콜입니다.

Multi-Agent 시스템에서는 Agent마다 역할, 도구, 실행 환경이 다를 수 있습니다. 서로 다른 프레임워크로 만든 Agent끼리도 협업할 수 있어야 하고, 단순 함수 호출보다 긴 작업, 상태 변화, 결과 산출물을 다룰 수 있는 구조가 필요합니다.

요청하는 Agent는 상대 Agent의 내부 구현을 몰라도 됩니다. 슬라이드 그림에서 배달앱 Agent는 배달관리 Agent와 가게 주문 관리 Agent에게 일을 맡기지만, 그 Agent들이 안에서 어떻게 만들어졌는지는 알 필요가 없습니다.

**왜 배우나:** 앞에서 만든 Multi-Agent는 Researcher와 Writer가 한 그래프 안에 함께 있었습니다. A2A를 쓰면 Researcher를 따로 떨어진 서버로 두고, 요청만 보내 결과를 받을 수 있습니다.

In [ ]:
show_quiz("../assets/quizzes/3-3/q11.json")

**정리:** A2A는 Agent끼리 일을 맡기고 결과를 주고받는 약속입니다. 맡기는 쪽은 상대의 내부 구현을 몰라도 됩니다.

### 1.2 MCP와 A2A의 차이 <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("3-3", [5])

MCP와 A2A는 모두 연결 표준이지만 연결하는 대상이 다릅니다.

| | 연결 대상 | 핵심 |
|---|---|---|
| MCP | Agent ↔ Tool·데이터 소스 | 도구 사용 |
| A2A | Agent ↔ Agent | 작업 위임과 협업 |

둘은 함께 쓸 수 있습니다. 전문 Agent는 A2A로 작업을 받고, 안에서는 MCP로 도구를 씁니다. 슬라이드 그림에서 개인 Agent는 가게 Agent에게 일을 맡기고(A2A), 가게 Agent는 재고 서비스와 POS 서비스를 MCP로 씁니다.

**왜 배우나:** 앞에서 MCP로 도구를 연결했습니다. 도구를 붙일지, 다른 Agent에게 맡길지 고를 때 이 구분이 기준이 됩니다.

In [ ]:
show_quiz("../assets/quizzes/3-3/q12.json")

**정리:** 도구를 쓰면 MCP, 다른 Agent에게 일을 맡기면 A2A입니다. 한 Agent가 둘을 함께 쓸 수 있습니다.

## 2. A2A 핵심 개념 <small>(슬라이드) (퀴즈)</small>

### 2.1 Agent Card <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("3-3", [6, 7])

Agent Card는 Agent가 자신이 누구이고 어떤 일을 할 수 있는지 외부에 설명하는 메타데이터입니다.

- Agent의 이름, 설명, 제공 기능을 담습니다.
- 입출력 모드, 연결 정보, 제공 Skill도 함께 설명합니다.
- 다른 Agent나 클라이언트는 이 카드를 보고 협업할 수 있는 상대인지 판단합니다.

슬라이드의 예시는 이름이 "Researcher Agent"이고, "웹 리서치"라는 Skill 하나를 가진 카드입니다.

In [ ]:
show_quiz("../assets/quizzes/3-3/q21.json")

**정리:** Agent Card는 Agent의 자기소개서입니다. 이름·설명·Skill·입출력 모드·연결 정보를 담습니다.

### 2.2 Task, Message, Artifact <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("3-3", [8])

A2A의 협업 단위는 셋으로 나눠 봅니다.

- **Task**: Agent에게 위임되는 작업의 단위
- **Message**: Agent끼리 주고받는 요청, 질문, 중간 응답
- **Artifact**: 보고서, 코드, 파일, 구조화 데이터 같은 최종 또는 중간 산출물

시간이 오래 걸리는 작업은 Task 상태와 Message 이력으로 어디까지 진행되었는지 추적합니다.

**왜 배우나:** 이번 시간의 실습 코드는 Task를 따로 다루지 않고 Message 한 번으로 답을 돌려줍니다. 작업이 길어지면 Task 상태와 Artifact로 진행과 결과를 나눠 관리한다는 점을 개념으로 알아 둡니다.

In [ ]:
show_quiz("../assets/quizzes/3-3/q22.json")

**정리:** Task는 맡긴 일, Message는 주고받는 말, Artifact는 만들어 낸 결과물입니다.

## 3. A2A Server 구현 <small>(슬라이드) (퀴즈)</small>

### 3.1 AgentExecutor <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("3-3", [9, 10])

AgentExecutor는 A2A 요청을 실제 Agent 실행으로 잇는 서버의 핵심 클래스입니다. `execute`에서 하는 일은 세 가지입니다.

1. `RequestContext`에서 사용자 입력과 task/context id를 읽습니다.
2. 내부 LangGraph Agent를 실행합니다(`ainvoke`).
3. 결과를 `t.Message`로 감싸 `EventQueue`에 발행합니다.

**왜 배우나:** 안쪽의 LangGraph Agent는 지금까지 만든 것과 같습니다. A2A는 그 바깥을 감싸 다른 Agent가 요청할 수 있게 해 주는 층입니다. 이미 만든 Agent를 A2A로 내놓을 때 새로 쓰는 부분이 바로 이 클래스입니다.

In [ ]:
show_quiz("../assets/quizzes/3-3/q31.json")

**정리:** AgentExecutor는 요청을 읽고, 내부 Agent를 실행하고, 결과를 EventQueue에 발행합니다.

### 3.2 AgentCard <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("3-3", [11])

코드에서는 `t.AgentCard`로 Agent Card를 만듭니다.

- 카드는 `/.well-known/agent-card.json` 주소에서 JSON으로 공개됩니다. 클라이언트는 이 주소를 읽어 Agent의 능력과 연결 정보를 알아냅니다.
- 이름, 설명, 입출력 모드, 지원 인터페이스(연결 주소)를 담습니다.
- `skills`에는 Agent가 제공하는 대표 기능을 적습니다.

In [ ]:
show_quiz("../assets/quizzes/3-3/q32.json")

**정리:** Agent Card는 정해진 주소 `/.well-known/agent-card.json`에 공개되어, 누구나 먼저 읽어 볼 수 있습니다.

### 3.3 RequestHandler <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("3-3", [12])

RequestHandler는 A2A 서버에서 요청을 받는 층(라우팅)과 Agent를 실행하는 층을 잇는 객체입니다.

- `agent_executor`에 실제 실행 로직(앞의 AgentExecutor)을 연결합니다.
- `task_store`와 `queue_manager`로 작업 상태와 이벤트를 관리합니다.
- Agent Card 조회 라우트와 JSON-RPC 라우트를 만들어 Starlette 앱에 등록합니다.

서버 쪽 세 요소를 합치면 이렇습니다. Agent Card는 "나는 누구인가"를 알리고, RequestHandler는 요청을 받아 넘기고, AgentExecutor는 실제로 Agent를 실행합니다.

In [ ]:
show_quiz("../assets/quizzes/3-3/q33.json")

**정리:** RequestHandler가 라우트와 AgentExecutor를 잇고, 작업 상태와 이벤트를 관리합니다.

## 4. A2A Client 구현 <small>(슬라이드) (퀴즈)</small>

### 4.1 A2ACardResolver <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("3-3", [13, 14])

클라이언트는 메시지를 보내기 전에 상대 Agent의 Card부터 읽습니다. 이 일을 `A2ACardResolver`가 맡습니다.

1. `A2A-Version` 헤더로 SDK 1.x 요청임을 밝힙니다.
2. Agent Card를 먼저 읽어 URL, 기능, 입출력 모드를 확인합니다.
3. 읽은 Card를 바탕으로 JSON-RPC transport와 Client를 만듭니다.

In [ ]:
show_quiz("../assets/quizzes/3-3/q41.json")

**정리:** 클라이언트 순서는 Card 조회 → transport·Client 생성입니다. 헤더로 버전을 밝힙니다.

### 4.2 SendMessageRequest <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("3-3", [15])

`SendMessageRequest`는 사용자 요청을 A2A Message로 감싸 원격 Agent에게 보내는 요청 객체입니다.

- 사용자 입력은 `t.Message`의 `parts`에 텍스트로 담습니다.
- `client.send_message(req)`는 응답을 하나씩 흘려보내는 스트림을 돌려줍니다.
- 응답 메시지의 `parts`에서 Agent가 만든 텍스트를 꺼냅니다.

서버가 결과를 `t.Message`로 발행했듯이, 클라이언트도 요청을 `t.Message`로 보냅니다. 주고받는 형식이 같습니다.

In [ ]:
show_quiz("../assets/quizzes/3-3/q42.json")

**정리:** 요청은 `t.Message`의 `parts`에 담아 보내고, 답도 응답 메시지의 `parts`에서 꺼냅니다.

## 5. 정리 퀴즈 <small>(퀴즈)</small>

In [ ]:
show_quiz("../assets/quizzes/3-3/final.json")

**이 노트북에서 본 것**
- A2A는 Agent가 다른 Agent에게 작업을 위임하고 결과를 주고받는 프로토콜입니다. 맡기는 쪽은 상대의 내부 구현을 몰라도 됩니다.
- MCP는 Agent와 도구를, A2A는 Agent와 Agent를 잇습니다. 한 Agent가 둘을 함께 쓸 수 있습니다.
- Agent Card는 이름·설명·Skill·입출력 모드·연결 정보를 담아 `/.well-known/agent-card.json`에 공개됩니다.
- 협업 단위는 Task(맡긴 일), Message(주고받는 말), Artifact(산출물)입니다.
- 서버: AgentExecutor가 내부 Agent를 실행해 결과를 EventQueue에 발행하고, RequestHandler가 라우트와 실행을 잇습니다.
- 클라이언트: Agent Card 조회 → Client 생성 → `t.Message`를 담은 SendMessageRequest 전송 → 응답의 `parts`에서 텍스트 꺼내기.